# Discovery técnico — ingestão IBGE por API

Este notebook faz parte do gate **Discovery** da feature `ibge-api-ingestion`.

## Objetivo

Entender empiricamente os contratos das APIs do IBGE que podem alimentar o Olist Customer Intelligence, com foco inicial em:

1. **API de Localidades** — cadastro e hierarquia territorial oficial;
2. **API SIDRA** — estatísticas agregadas;
3. **População estimada por município** — primeira Bronze candidata;
4. **PIB dos municípios** — segunda Bronze candidata e teste de reutilização do desenho.

> Este notebook é deliberadamente exploratório. Ele **não** implementa ainda as classes de produção.
> A saída esperada é conhecimento suficiente para fechar requisitos e desenho técnico.

## Perguntas que queremos responder

Durante a execução, observe especialmente:

- Como a API identifica municípios?
- O código municipal é estável e suficiente como chave de integração?
- Como a API de Localidades representa UF, região e região intermediária/imediata?
- Como uma consulta SIDRA é composta?
- Qual é o formato real do payload SIDRA?
- A primeira linha do SIDRA é metadado/cabeçalho?
- Como período, variável, unidade territorial e território aparecem na resposta?
- Quais símbolos especiais aparecem em `Valor`?
- Conseguimos obter todos os municípios em uma única chamada?
- O mesmo mecanismo de consulta serve para população e PIB?
- Há sinais de limites de volume ou necessidade de chunking?

In [ ]:
from __future__ import annotations

from collections import Counter
from pprint import pprint
from typing import Any

import requests

TIMEOUT_SECONDS = 30

LOCALITIES_BASE_URL = "https://servicodados.ibge.gov.br/api/v1/localidades"
SIDRA_BASE_URL = "https://apisidra.ibge.gov.br/values"

POPULATION_TABLE_ID = 6579
POPULATION_VARIABLE_ID = 9324
MUNICIPAL_LEVEL_ID = 6

# São Paulo/SP — usado apenas para chamadas pequenas durante o discovery.
SAMPLE_MUNICIPALITY_ID = 3550308

## 1. Helper HTTP mínimo

No código de produção já existe um `APIClient` genérico com retry, timeout, logging e tratamento de erros.

Aqui usamos `requests` diretamente para deixar a exploração transparente e evitar que comportamento da nossa abstração esconda detalhes do contrato externo.

In [ ]:
def get_json(url: str) -> dict[str, Any] | list[Any]:
    response = requests.get(url, timeout=TIMEOUT_SECONDS)
    print("status_code:", response.status_code)
    print("content_type:", response.headers.get("content-type"))
    print("url:", response.url)

    response.raise_for_status()
    return response.json()

# Parte A — API de Localidades

A API de Localidades fornece a estrutura territorial oficial do IBGE.

Primeiro vamos consultar **um município específico** para entender o shape sem trazer o Brasil inteiro.

In [ ]:
sample_locality_url = (
    f"{LOCALITIES_BASE_URL}/municipios/{SAMPLE_MUNICIPALITY_ID}"
)

sample_locality = get_json(sample_locality_url)
pprint(sample_locality)

## Inspeção do contrato de município

A célula abaixo lista as chaves de primeiro nível e navega pelos objetos territoriais aninhados.

O objetivo não é decidir ainda o schema final, mas identificar quais atributos podem ser úteis para uma dimensão geográfica.

In [ ]:
print("top_level_keys:", sorted(sample_locality.keys()))

for key, value in sample_locality.items():
    print(f"\n{key}: {type(value).__name__}")
    if isinstance(value, dict):
        print("  nested_keys:", sorted(value.keys()))

## Todos os municípios

Agora trazemos a lista nacional. Isso permite verificar volume, unicidade dos códigos e distribuição por UF.

A chamada é pequena o suficiente para discovery e ajuda a avaliar se precisamos de paginação ou chunking na API de Localidades.

In [ ]:
all_municipalities_url = f"{LOCALITIES_BASE_URL}/municipios?orderBy=nome"
municipalities = get_json(all_municipalities_url)

print("municipality_count:", len(municipalities))
print("first_record:")
pprint(municipalities[0])

In [ ]:
municipality_ids = [item["id"] for item in municipalities]

print("unique_ids:", len(set(municipality_ids)))
print("duplicates:", len(municipality_ids) - len(set(municipality_ids)))

uf_counts = Counter(
    item["regiao-imediata"]["regiao-intermediaria"]["UF"]["sigla"]
    for item in municipalities
    if item.get("regiao-imediata")
)

print("\nmunicipalities_by_uf:")
pprint(dict(sorted(uf_counts.items())))

### Ponto de atenção

A estrutura territorial pode conter campos nulos ou mudar conforme o tipo de unidade territorial.
Não devemos assumir no código de produção que toda navegação aninhada é sempre preenchida sem validar o contrato real.

# Parte B — API SIDRA

A API SIDRA expõe dados estatísticos através de segmentos na URL.

Para este discovery usaremos:

- `t/{id}` → tabela;
- `n6/{território}` → nível territorial 6 = município;
- `v/{id}` → variável;
- `p/{período}` → período;
- `last 1` → último período disponível.

A tabela **6579** corresponde a **População residente estimada** e a variável **9324** à população estimada.

## 2. Primeira consulta SIDRA — um município, último período

Começamos com uma resposta mínima para entender o payload.

O SIDRA costuma devolver uma lista na qual o primeiro item descreve os campos e os itens seguintes contêm os dados.
Vamos confirmar isso empiricamente.

In [ ]:
population_sample_url = (
    f"{SIDRA_BASE_URL}/t/{POPULATION_TABLE_ID}"
    f"/n{MUNICIPAL_LEVEL_ID}/{SAMPLE_MUNICIPALITY_ID}"
    f"/v/{POPULATION_VARIABLE_ID}"
    "/p/last%201"
    "?formato=json"
)

population_sample = get_json(population_sample_url)

print("items:", len(population_sample))
pprint(population_sample)

## Entendendo os códigos de coluna do SIDRA

O SIDRA usa chaves compactas (`NC`, `NN`, `V`, `D1C`, `D1N` etc.).
Em vez de codificarmos significado por memória, podemos usar o primeiro item retornado como mapa de cabeçalho durante o discovery.

In [ ]:
sidra_header = population_sample[0]
sidra_rows = population_sample[1:]

print("header:")
pprint(sidra_header)

print("\nfirst_data_row:")
pprint(sidra_rows[0])

In [ ]:
def decode_sidra_row(
    header: dict[str, str],
    row: dict[str, str],
) -> dict[str, str]:
    return {
        header.get(key, key): value
        for key, value in row.items()
    }


decoded_sample = [
    decode_sidra_row(sidra_header, row)
    for row in sidra_rows
]

pprint(decoded_sample)

## 3. População — todos os municípios, último período

Esta é a consulta mais próxima da primeira Bronze planejada.

O objetivo agora é verificar:

- quantidade de registros;
- período retornado;
- unicidade por município;
- códigos territoriais;
- presença de valores especiais;
- tamanho aproximado da resposta.

In [ ]:
population_all_url = (
    f"{SIDRA_BASE_URL}/t/{POPULATION_TABLE_ID}"
    f"/n{MUNICIPAL_LEVEL_ID}/all"
    f"/v/{POPULATION_VARIABLE_ID}"
    "/p/last%201"
    "?formato=json"
)

population_payload = get_json(population_all_url)

population_header = population_payload[0]
population_rows = population_payload[1:]

print("data_rows:", len(population_rows))
print("header:")
pprint(population_header)
print("\nfirst_3_rows:")
pprint(population_rows[:3])

In [ ]:
# Descobrir quais campos representam município, período, variável e valor
# sem assumir antecipadamente os códigos D1/D2/D3.

decoded_population_preview = [
    decode_sidra_row(population_header, row)
    for row in population_rows[:3]
]

pprint(decoded_population_preview)

## Valores especiais

No SIDRA, `Valor` pode ser numérico ou um símbolo estatístico, por exemplo dados indisponíveis, não aplicáveis ou inibidos.

Na Bronze devemos preservar o valor recebido. Conversão numérica e tratamento semântico pertencem a uma etapa posterior, salvo decisão explícita no design.

In [ ]:
value_key = "V"

value_counts = Counter(row.get(value_key) for row in population_rows)
non_numeric_values = sorted(
    value
    for value in value_counts
    if value is not None
    and not value.replace(".", "", 1).replace("-", "", 1).isdigit()
)

print("distinct_values:", len(value_counts))
print("non_numeric_values:", non_numeric_values[:50])

## Unicidade territorial

Vamos localizar dinamicamente no header qual coluna contém `Município (Código)` e verificar duplicidades no recorte de um único período/variável.

In [ ]:
municipality_code_key = next(
    (
        key
        for key, label in population_header.items()
        if "Município (Código)" in label
    ),
    None,
)

print("municipality_code_key:", municipality_code_key)

if municipality_code_key:
    codes = [row[municipality_code_key] for row in population_rows]
    print("rows:", len(codes))
    print("unique_municipalities:", len(set(codes)))
    print("duplicate_rows:", len(codes) - len(set(codes)))

## 4. Múltiplos períodos

Uma ingestão histórica precisa funcionar para mais de um ano.
Vamos consultar poucos períodos para entender como o período aparece no payload e confirmar a cardinalidade esperada `município × período`.

In [ ]:
population_history_sample_url = (
    f"{SIDRA_BASE_URL}/t/{POPULATION_TABLE_ID}"
    f"/n{MUNICIPAL_LEVEL_ID}/{SAMPLE_MUNICIPALITY_ID}"
    f"/v/{POPULATION_VARIABLE_ID}"
    "/p/2019,2020,2021,2024,2025"
    "?formato=json"
)

population_history_sample = get_json(population_history_sample_url)

history_header = population_history_sample[0]
history_rows = population_history_sample[1:]

pprint([
    decode_sidra_row(history_header, row)
    for row in history_rows
])

### Observação sobre períodos

A série de estimativas não deve ser tratada como uma sequência anual perfeita sem consultar os períodos realmente disponíveis na tabela.

Para produção, o cliente deve receber períodos explicitamente ou suportar seletores SIDRA (`last`, `all`) sem fabricar anos inexistentes.

# Parte C — PIB dos Municípios

Agora testamos a hipótese mais importante para a arquitetura:

> O mesmo mecanismo genérico de consulta SIDRA consegue explorar outra tabela sem criar um cliente específico de domínio?

A tabela **5938** contém PIB municipal e valor adicionado por atividade econômica.

Nesta primeira chamada pedimos `v/all` para **um único município e um único período**. Isso serve para descobrir os identificadores de variáveis disponíveis sem gerar uma resposta nacional grande.

In [ ]:
PIB_TABLE_ID = 5938

pib_sample_url = (
    f"{SIDRA_BASE_URL}/t/{PIB_TABLE_ID}"
    f"/n{MUNICIPAL_LEVEL_ID}/{SAMPLE_MUNICIPALITY_ID}"
    "/v/all"
    "/p/last%201"
    "?formato=json"
)

pib_sample = get_json(pib_sample_url)

pib_header = pib_sample[0]
pib_rows = pib_sample[1:]

print("data_rows:", len(pib_rows))
print("header:")
pprint(pib_header)
print("\ndecoded_rows:")
pprint([
    decode_sidra_row(pib_header, row)
    for row in pib_rows
])

## Identificadores de variáveis do PIB

A célula abaixo procura no header a coluna cujo rótulo é `Variável (Código)` e monta o catálogo observado na chamada.

Esses IDs poderão virar configuração de domínio posteriormente; eles **não** devem ficar espalhados em strings de URL.

In [ ]:
variable_code_key = next(
    (
        key
        for key, label in pib_header.items()
        if "Variável (Código)" in label
    ),
    None,
)

variable_name_key = next(
    (
        key
        for key, label in pib_header.items()
        if label == "Variável"
    ),
    None,
)

print("variable_code_key:", variable_code_key)
print("variable_name_key:", variable_name_key)

if variable_code_key and variable_name_key:
    variables = {
        row[variable_code_key]: row[variable_name_key]
        for row in pib_rows
    }
    pprint(variables)

# Parte D — compatibilidade com o `APIClient` do projeto

Até aqui exploramos o contrato externo diretamente.

Agora validamos apenas se a infraestrutura HTTP existente consegue executar uma chamada IBGE sem mudança estrutural. Isso **não** é ainda a implementação de `SidraClient`.

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise RuntimeError("Repository root not found.")


repo_root = find_repo_root(Path.cwd())
src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from olist_data_platform.platform.http.api_client import APIClient

In [ ]:
localities_client = APIClient(
    base_url=LOCALITIES_BASE_URL,
    timeout=TIMEOUT_SECONDS,
)

project_client_sample = localities_client.get(
    f"municipios/{SAMPLE_MUNICIPALITY_ID}"
)

pprint(project_client_sample)

# Parte E — conclusões a registrar após a execução

Ao terminar o notebook, registre abaixo as conclusões observadas. Não preencha por expectativa; use somente o que as células realmente retornarem.

## Contrato de Localidades

- Quantidade de municípios:
- Chave territorial:
- Campos úteis:
- Campos opcionais/nulos:
- Paginação necessária?:

## Contrato SIDRA

- Estrutura do primeiro item:
- Estrutura das linhas de dados:
- Campo de valor:
- Campo de período:
- Campo de município:
- Comportamento de `last 1`:
- Valores especiais encontrados:
- Volume da consulta nacional:
- Limite/chunking observado:

## População

- Tabela:
- Variável:
- Períodos disponíveis observados:
- Cardinalidade esperada:
- `dt_base` candidata:

## PIB

- Tabela:
- Variáveis observadas:
- Mesmo padrão de consulta funciona?:

## Decisões que o discovery deverá suportar

1. contrato de `SidraQuery`;
2. responsabilidades de `SidraClient`;
3. responsabilidades de `IbgeLocalitiesClient`;
4. estratégia de parsing/preservação da Bronze;
5. estratégia de períodos e reprocessamento;
6. necessidade ou não de chunking por UF/município/período.

# Próximo gate

Depois de executar e revisar as evidências deste notebook, seguimos para **Requisitos**.

Não implementar as classes de produção antes de fechar:

- contrato de consulta;
- comportamento de erro;
- schema mínimo da primeira Bronze;
- semântica de `dt_base`;
- idempotência/reprocessamento;
- testes necessários.